# Preliminary: Analysis of the cosmology baseline $F$ problems

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

In [ ]:
feature = sympy.symbols("feature")
x1, x2 = sympy.symbols("x1:3")
x = sympy.abc.x
y = sympy.abc.y

## Load data

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report.run_set.unique()

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
f_indices = full_report.data_set.apply(lambda x: x.startswith("F"))
fr2 = full_report.loc[f_indices].set_index(["run_set", "data_set", "sample_num"])

In [ ]:
fr2["sympy"] = fr2.expr_original_syms.apply(au.parse_if_needed)
fr2["sympy_defuzz"] = fr2.expr_original_syms_defuzz.apply(au.parse_if_needed)

In [ ]:
fr2

In [ ]:
srb3_key = "SRB-2026-07-20-1300"
cht4_key = "CHT-2026-07-20-1300"
srb3 = fr2.loc[srb3_key]
cht4 = fr2.loc[cht4_key]

In [ ]:
(fr2.groupby(level=["run_set","data_set"]).size() == 32).all()

These are the best ones overall

In [ ]:
srb3_min_mse_ixs = srb3.groupby(level=["data_set"]).mse.idxmin()
cht4_min_mse_ixs = cht4.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb3.loc[srb3_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht4.loc[cht4_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

To be consistent with all the other runs, I need to use `srb3` and `cht4` here.

In [ ]:
srb_key = srb3_key
srb = srb3
srb_min_mse_ixs = srb3_min_mse_ixs
cht_key = cht4_key
cht = cht4
cht_min_mse_ixs = cht4_min_mse_ixs

## Polynomials

Generally, the polynomial problems are easy.

In [ ]:
data_sets_polynomial = ["F1", "F4"]

The best ones are exactly correct.

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_polynomial, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
au.count_by_threshold(srb.loc[data_sets_polynomial])

With some generous defuzzing, SRB gets all of the $F_1$'s exactly.

In [ ]:
(srb.loc["F1"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-3)))

In [ ]:
(cht.loc["F1"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-3)))

In [ ]:
srb.loc["F4"]

In [ ]:
srb.loc["F4",18]

All of the $F_1$ solutions are polynomials.

In [ ]:
srb.loc["F1"].sympy_defuzz.apply(lambda e: e.is_polynomial(feature)).sum()

Most of the $F_4$ solutions are polynomials

In [ ]:
srb.loc["F4"].sympy_defuzz.apply(lambda e: e.is_polynomial(x1, x2)).sum()

It looks like all of the $F_4$'s are essentially correct.

In [ ]:
(srb.loc["F4"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-2)))

With CHT, looks like 20 or so are perfectly correct.

In [ ]:
(cht.loc["F4"]
    .sympy_defuzz
    .apply(lambda e:
           au.replace_near_integer(sympy.expand(e),
                                   tolerance=5e-3)))

In [ ]:
polynomial_plot_params = {
    "spiffy_titles": ["$F_1$","$F_4$"],
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 299),
    "mse_lims": (1e-30, 1e3),
    "complexity_binwidth": 20,
    "mse_binwidth": 2.0,
    "xlabel": "Complexity (defuzzed)",
    "ylabel": "MSE",
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_polynomial],
    file_stem="srb-polynomial-complexit-mse-displot",
    **polynomial_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_polynomial],
    file_stem="cht-polynomial-complexity-mse-displot",
    **polynomial_plot_params
)

## Rational functions

In [ ]:
data_sets_rational = ["F7"]

In [ ]:
srb.loc[srb_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht.loc[cht_min_mse_ixs].loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
au.count_by_threshold(srb.loc[data_sets_rational])

In [ ]:
au.count_by_threshold(cht.loc[data_sets_rational])

In [ ]:
srb.loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb.loc["F7"].sympy_defuzz.apply(
    lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-4))

In [ ]:
cht.loc[data_sets_rational, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht.loc["F7"].sympy_defuzz.apply(
    lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-4))

In [ ]:
rational_plot_params = {
    "spiffy_titles": ["$F_7$"],
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 149),
    "mse_lims": (1e-30, 1e7),
    "complexity_binwidth": 10,
    "mse_binwidth": 2.0,
    "xlabel": "Complexity (defuzzed)",
    "ylabel": "MSE",
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_rational],
    file_stem="srb-rational-complexity-mse-displot",
    **rational_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_rational],
    file_stem="cht-rational-complexity-mse-displot",
    **rational_plot_params
)

## Medium: $F_5$

In [ ]:
data_sets_medium = ["F5"]

In [ ]:
srb.loc["F5"]

In [ ]:
srb.loc["F5"].sympy_defuzz.apply(
    lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-4))

My usual cheats don't help here, because I use a larger exponential and trig inventory, which leads to a lot of distraction and cruft that really damages the solutions.

In [ ]:
cht.loc["F5"]

In [ ]:
cht.loc["F5"].sympy_defuzz.apply(
    lambda e: au.replace_near_integer(sympy.expand(e), tolerance=1.0e-4))

In [ ]:
medium_plot_params = {
    "spiffy_titles": ["$F_5$"],
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 79),
    "mse_lims": (1e-30, 1e1),
    "complexity_binwidth": 5,
    "mse_binwidth": 2.0,
    "xlabel": "Complexity (defuzzed)",
    "ylabel": "MSE",
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[data_sets_medium],
    file_stem="srb-medium-complexity-mse-displot",
    **medium_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[data_sets_medium],
    file_stem="cht-medium-complexity-mse-displot",
    **medium_plot_params
)

## Hard: $F_2$

None of these are any good.

In [ ]:
srb.loc["F2"]

In [ ]:
cht.loc["F2"]

## Hard: $F_8$

A bunch of these are correct

In [ ]:
srb.loc["F8", ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht.loc["F8", ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
f8_plot_params = {
    "spiffy_titles": ["$F_8$"],
    "complexity_col": "complexity_defuzz",
    "complexity_lims": (0, 139),
    "mse_lims": (1e-30, 1e7),
    "complexity_binwidth": 10,
    "mse_binwidth": 2.0,
    "xlabel": "Complexity (defuzzed)",
    "ylabel": "MSE",
}

In [ ]:
au.complexity_mse_displot(
    srb.loc[["F8"]],
    file_stem="srb-f8-complexity-mse-displot",
    **f8_plot_params
)

In [ ]:
au.complexity_mse_displot(
    cht.loc[["F8"]],
    file_stem="cht-f8-complexity-mse-displot",
    **f8_plot_params
)